# Chapter 05 트리 알고리즘
## 05-3 트리의 앙상블
### 정형 데이터와 비정형 데이터
* 4장까지는 생선의 길이, 높이, 무게 등을 데이터로 사용. 이 데이터는 CSV 파일에 가지런히 정리되어 있었음. 또 이번 장에서 사용한 와인 데이터도 CSV 파일. -> 이런 형태의 데이터를 **정형 데이터structured data**라고 부름.
* 이와 반대되는 데이터를 **비정형 데이터unstructured data**
  * 텍스트 데이터, 디지털카메라로 찍은 사진, 핸드폰으로 듣는 디지털 음악 등

> 텍스트나 사진을 데이터베이스에 저장할 수는 없나요?
> * 아님, 저장할 수도 있음. 다만 여기에서는 보편적인 사례를 설명.

* 지금까지 배운 머신러닝 알고리즘은 정형 데이터에 잘 맞음. 그중에 정형 데이터를 다루는 데 가장 뛰어난 성과를 내는 알고리즘이 **앙상블 학습ensemble learning**. 이 알고리즘은 대부분 결정 트리를 기반으로 만들어져 있음.
* 그럼 비정형 데이터에는 어떤 알고리즘을 사용해야 할까? -> 바로 7장에서 배울 신경망 알고리즘. 비정형 데이터는 규칙성을 찾기 어려워 전통적인 머신러닝 방법으로는 모델을 만들기 까다로움. 하지만 신경망 알고리즘의 놀라운 발전 덕분에 사진을 인식하고 텍스트를 이해하는 모델을 만들 수 있음.

### 랜덤 포레스트
* **랜덤 포레스트Random Forest**는 앙상블 학습의 대표 주자 중 하나로 안정적인 성능 덕분에 널리 사용되고 있음. 앙상블 학습을 적용할 때 가장 먼저 랜덤 포레스트를 시도해 보길 권함.
* 랜덤 포레스트는 결정 트리를 랜덤하게 만들어 결정 트리(나무)의 숲을 만듦. 그리고 각 결정 트리의 예측을 사용해 최종 예측을 만듦.
* 먼저 랜덤 포레스트는 각 트리를 훈련하기 위한 데이터를 랜덤하게 만드는데, 이 데이터를 만드는 방법이 독특. 우리가 입력한 훈련 데이터에서 랜덤하게 샘플을 추출하여 훈련 데이터를 만듦. 이때 한 샘플이 중복되어 추출될 수도 있음.
* 예를 들어 1,000개의 샘플이 들어있는 가방에서 100개의 샘플을 뽑는다면 먼저 1개를 뽑고, 뽑았던 1개를 다시 가방에 넣음. 이런 식으로 계속해서 100개를 가방에서 뽑으면 중복된 샘플을 뽑을 수 있음. -> **부트스트랩 샘플bootstrap sample**이라고 부름.
* 기본적으로 부트스트랩 샘플은 훈련 세트의 크기와 같게 만듦.

> 부트스트랩이 뭔가요?
> * 보통 부트스트랩 방식이라고 하는데, 데이터 세트에서 중복을 허용하여 데이터를 샘플링하는 방식을 의미. 본문에서 설명한 것처럼 가방에 1,000개의 샘플이 있을 때 먼저 1개를 뽑고, 다시 가방에 넣어 그다음 샘플을 뽑는 방식을 뜻함. 부트스트랩 샘플이란 결국 부트스트랩 방식으로 샘플링하여 분류한 데이터라는 의미.

* 또한 각 노드를 분할할 때 전체 특성 중에서 일부 특성을 무작위로 고른 다음 이 중에서 최선의 분할을 찾음. 분류 모델인 RandomForestClassifier는 기본적으로 전체 특성 개수의 제곱근만큼의 특성을 선택.
* 다만 회귀 모델인 RandomForestRegressor는 전체 특성을 사용.
* 사이킷런의 랜덤 포레스트는 기본적으로 100개의 결정 트리를 이런 방식으로 훈련. 그다음 분류일 때는 각 트리의 클래스별 확률을 평균하여 가장 높은 확률을 가진 클래스를 예측으로 삼음. 회귀일 때는 단순히 각 트리의 예측을 평균.
* 랜덤 포레스트는 랜덤하게 선택한 샘플과 특성을 사용하기 때문에 훈련 세트에 과대적합되는 것을 막아주고 검증 세트와 테스트 세트에서 안정적인 성능을 얻을 수 있음. 종종 기본 매개변수 설정만으로도 아주 좋은 결과를 냄.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
wine = pd.read_csv('https://bit.ly/wine_csv_data')
data = wine[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine['class'].to_numpy()
train_input, test_input, train_target, test_target = train_test_split(
    data, target, test_size=0.2, random_state=42
)

In [3]:
from sklearn.model_selection import cross_validate
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_jobs=-1, random_state=42)
scores = cross_validate(rf, train_input, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9973541965122431 0.8905151032797809


* 출력된 결과를 보면 훈련 세트에 다소 과대적합된 것 같음.

In [ ]:
rf.fit(train_input, train_target) # 앞의 랜덤 포레스트 모델을 훈련 세트에 훈련한 후
print(rf.feature_importances_) # 특성 중요도를 출력

* 이 결과를 앞의 1절 '결정 트리'에서 만든 특성 중요도234쪽와 비교해 볼 것.
* 각각 [알코올 도수, 당도, pH]였는데, 두 번째 특성인 당도의 중요도가 감소하고 알코올 도수와 pH 특성의 중요도가 조금 상승. 이런 이유는 랜덤 포레스트가 특성의 일부를 랜덤하게 선택하여 결정 트리를 훈련하기 때문. 그 결과 하나의 특성에 과도하게 집중하지 않고 좀 더 많은 특성이 훈련에 기여할 기회를 얻음. 이는 과대적합을 줄이고 일반화 성능을 높이는 데 도움이 됨.
* RandomForestClassifier에는 재미있는 기능이 하나 더 있는데, 자체적으로 모델을 평가하는 점수를 얻을 수 있음. 랜덤 포레스트에는 훈련 세트에서 중복을 허용하여 부트스트랩 샘플을 만들어 결정 트리를 훈련한다고 했음. 이때 부트스트랩 샘플에 포함되지 않고 남는 샘플이 있음. 이런 샘플을 **OOBout of bag 샘플**이라고 함. 이 남는 샘플을 사용하여 부트스트랩 샘플로 훈련한 결정 트리를 평가할 수 있음. 마치 검증 세트의 역할!

In [5]:
rf = RandomForestClassifier(oob_score=True, n_jobs=-1, random_state=42)
rf.fit(train_input, train_target)
print(rf.oob_score_)

0.8934000384837406


* 교차 검증에서 얻은 점수와 매우 비슷한 결과를 얻었음. OOB 점수를 사용하면 교차 검증을 대신할 수 있어서 결과적으로 훈련 세트에 더 많은 샘플을 사용할 수 있음.

### 엑스트라 트리
* 기본적으로 100개의 결정 트리를 훈련. 랜덤 포레스트와 동일하게 결정 트리가 제공하는 대부분의 매개변수를 지원. 또한 전체 특성 중에 일부 특성을 랜덤하게 선택하여 노드를 분할하는 데 사용.
* 랜덤 포레스트와 엑스트라 트리의 차이점은 부트스트랩 샘플을 사용하지 않는다는 점. 즉 각 결정 트리를 만들 때 전체 훈련 세트를 사용. 대신 노드를 분할할 때 가장 좋은 분할을 찾는 것이 아니라 무작위로 분할!
* 2절의 확인 문제에서 DecisionTreeClassifier의 splitter 매개변수를 'random'으로 지정했는데, 엑스트라 트리가 사용하는 결정 트리가 바로 splitter='random'인 결정 트리.
* 하나의 결정 트리에서 특성을 무작위로 분할한다면 성능이 낮아지겠지만 많은 트리를 앙상블 하기 때문에 과대적합을 막고 검증 세트의 점수를 높이는 효과가 있음.

In [7]:
from sklearn.ensemble import ExtraTreesClassifier
et = ExtraTreesClassifier(n_jobs=-1, random_state=42)
scores = cross_validate(et, train_input, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9974503966084433 0.8887848893166506


* 랜덤 포레스트와 비슷한 결과를 얻음.
* 보통 엑스트라 트리가 무작위성이 좀 더 크기 때문에 랜덤 포레스트보다 더 많은 결정 트리를 훈련해야 함. 하지만 랜덤하게 노드를 분할하기 때문에 빠른 계산 속도가 엑스트라 트리의 장점.

In [8]:
et.fit(train_input, train_target)
print(et.feature_importances_)

[0.20183568 0.52242907 0.27573525]


### 그레이디언트 부스팅
* **그레이디언트 부스팅gradient boosting**은 깊이가 얕은 결정 트리를 사용하여 이전 트리의 오차를 보완하는 방식으로 앙상블 하는 방법
* 사이킷런의 GradientBoostingClassifier는 기본적으로 깊이가 3인 결정 트리를 100개 사용. 깊이가 얕은 결정 트리를 사용하기 때문에 과대적합에 강하고 일반적으로 높은 일반화 성능을 기대할 수 있음.
* 4장에서 배웠던 경사 하강법200쪽을 사용하여 트리를 앙상블에 추가. 분류에선느 로지스틱 손실 함수를 사용하고 회귀에서는 평균 제곱 오차 함수를 사용.
* 4장에서 경사 하강법은 손실 함수를 산으로 정의하고 가장 낮은 곳을 찾아 내려오는 과정으로 설명. 이때 가장 낮은 곳을 찾아 내려오는 방법은 모델의 가중치와 절편을 조금씩 바꾸는 것. 그레이디언트 부스팅은 결정 트리를 계속 추가하면서 가장 낮은 곳을 찾아 이동.

In [10]:
from sklearn.ensemble import GradientBoostingClassifier
gb = GradientBoostingClassifier(random_state=42)
scores = cross_validate(gb, train_input, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.8881086892152563 0.8720430147331015


* 거의 과대적합이 되지 않음. 그레이디언트 부스팅은 결정 트리의 개수를 늘려도 과대적합에 매우 강함. 학습률을 증가시키고 트리의 개수를 늘리면 조금 더 성능이 향상될 수 있음.

In [12]:
gb = GradientBoostingClassifier(n_estimators=500, learning_rate=0.2, random_state=42)
scores = cross_validate(gb, train_input, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.9464595437171814 0.8780082549788999


* 결정 트리 개수를 500개로 5배나 늘렸지만 과대적합을 잘 억제하고 있음. 학습률 learning_rate의 기본값은 0.1. 그레이디언트 부스팅도 특성 중요도를 제공. 결과에서 볼 수 있듯이 그레이디언트 부스팅이 랜덤 포레스트보다 일부 특성(당도)에 더 집중함.

In [13]:
gb.fit(train_input, train_target)
print(gb.feature_importances_)

[0.15887763 0.6799705  0.16115187]


* 재미있는 매개변수가 하나 있음. 트리 훈련에 사용할 훈련 세트의 비율을 정하는 subsample. 이 매개변수의 기본값은 1.0으로 전체 훈련 세트를 사용. 하지만 subsample이 1보다 작으면 훈련 세트의 일부를 사용. 이는 마치 경사 하강법 단계마다 일부샘플을 랜덤하게 선택하여 진행하는 확률적 경사 하강법이나 미니배치 경사 하강법과 비슷.
* 일반적으로 그레이디언트 부스팅이 랜덤 포레스트보다 조금 더 높은 성능을 얻을 수 있음. 하지만 순서대로 트리를 추가하기 때문에 훈련 속도가 느림. 즉 GradientBoostingClassifier에는 n_jobs 매개변수가 없음. 그레이디언트 부스팅의 회귀 버전은 GradientBoostingRegressor. 그레이디언트 부스팅의 속도와 성능을 더욱 개선한 것이 다음에 살펴볼 히스토그램 기반 그레이디언트 부스팅.

### 히스토그램 기반 그레이디언트 부스팅
* **히스토그램 기반 그레이디언트 부스팅Histogram-based Gradient Boosting**은 정형 데이터를 다루는 머신러닝 알고리즘 중에 가장 인기가 높은 알고리즘. 히스토그램 기반 그레이디언트 부스팅은 먼저 입력 특성을 256개의 구간으로 나눔. 따라서 노드를 분할할 때 최적의 분할을 매우 빠르게 찾을 수 있음. 히스토그램 기반 그레이디언트 부스팅은 256개의 구간 중에서 하나를 떼어 놓고 누락된 값을 위해서 사용. 따라서 입력에 누락된 특성이 있더라도 이를 따로 전처리할 필요가 없음!
* 사이킷런의 히스토그램 기반 그레이디언트 부스팅 클래스는 HistGradientBoostingClassifier. 일반적으로 HistGradientBoostingClassifier는 기본 매개변수에서 안정적인 성능을 얻을 수 있음. HistGradientBoosingClassifier에는 트리의 개수를 지정하는데 n_estimators 대신에 부스팅 반복 횟수를 지정하는 max_iter를 사용. 성능을 높이려면 max_iter 매개변수를 테스트해 볼 것.

In [14]:
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingClassifier
hgb = HistGradientBoostingClassifier(random_state=42)
scores = cross_validate(hgb, train_input, train_target, return_train_score=True)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

/usr/local/lib/python3.13/dist-packages/sklearn/experimental/enable_hist_gradient_boosting.py:19: UserWarning: Since version 1.0, it is not needed to import enable_hist_gradient_boosting anymore. HistGradientBoostingClassifier and HistGradientBoostingRegressor are now stable and can be normally imported from sklearn.ensemble.
  warnings.warn(


0.9321723946453317 0.8801241948619236


* 과대적합을 잘 억제하면서 그레이디언트 부스팅보다 조금 더 높은 성능을 제공.
* 히스토그램 기반 그레이디언트 부스팅의 특성 중요도를 계산하기 위해 permutation_importance() 함수를 사용. 이 함수는 특성을 하나씩 랜덤하게 섞어서 모델의 성능이 변화하는지를 관찰하여 어떤 특성이 중요한지를 계산. 훈련 세트뿐만 아니라 테스트 세트에도 적용할 수 있고 사이킷런에서 제공하는 추정기 모델에 모두 사용할 수 있음.

In [15]:
from sklearn.inspection import permutation_importance

hgb.fit(train_input, train_target)
result = permutation_importance(hgb, train_input, train_target, n_repeats=10, random_state=42, n_jobs=-1)
print(result.importances_mean)

[0.08876275 0.23438522 0.08027708]


In [16]:
result = permutation_importance(hgb, test_input, test_target, n_repeats=10, random_state=42, n_jobs=-1)
print(result.importances_mean)

[0.05969231 0.20238462 0.049     ]


* 테스트 세트의 결과를 보면 그레이디언트 부스팅과 비슷하게 조금 더 당도에 집중하고 있다는 것을 알 수 있음. 이런 분석을 통해 모델을 실전에 투입했을 때 어떤 특성에 관심을 둘지 예상할 수 있음.

In [17]:
hgb.score(test_input, test_target)

0.8723076923076923

* 테스트 세트에서는 약 87%의 정확도를 얻었음. 실전에 투입하면 성능은 이보다는 조금 더 낮을 것. 앙상블 모델은 확실히 단일 결정트리보다 좋은 결과를 얻을 수 있음!
* 사이킷런 말고도 그레이디언트 부스팅 알고리즘을 구현한 라이브러리가 여럿 있음.
* 가장 대표적인 라이브러리는 XGBoost. 놀랍게도 이 라이브러리도 코랩에서 사용할 수 있을 뿐만 아니라 사이킷런의 cross_validate() 함수와 함께 사용할 수도 있음. XGBoost는 다양한 부스팅 알고리즘을 지원. tree_method 매개변수를 'hist'로 지정하면 히스토그램 기반 그레이디언트 부스팅을 사용할 수 있음.

In [ ]:
from xgboost import XGBClassifier
xgb = XGBClassifier(tree_methos='hist', random_state=42)
scores = cross_validate(xgb, train_input, train_target, return_train_score=True)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

* 널리 사용하는 또 다른 히스토그램 기반 그레이디언트 부스팅 라이브러리는 마이크로소프트에서 만든 LightGBM. LightGBM은 빠르고 최신 기술을 많이 적용하고 있어 인기가 점점 높아지고 있음. LightBGM도 코랩에 이미 설치되어 있어 바로 테스트해 볼 수 있음!

In [20]:
from lightgbm import LGBMClassifier
lgb = LGBMClassifier(random_state=42)
scores = cross_validate(lgb, train_input, train_target, return_train_score=True, n_jobs=-1)
print(np.mean(scores['train_score']), np.mean(scores['test_score']))

0.935828414851749 0.8801251203079884
